# Life Quality Index and SWTP

This example follows the simple JCSS LQI resistance-demand calculation: a design parameter changes the resistance, FORM estimates the failure probability, and a cost-benefit objective is checked together with an LQI/SWTP life-safety target.

The LQI helpers in Pystra are post-processing tools. They do not change the stochastic model or the reliability method; they use the `pf` and `beta` results returned by FORM, SORM, or simulation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pystra as ra

pd.options.display.float_format = "{:,.4g}".format

## Reliability model

The limit state is written as

$$g(f_y, A_s, S) = f_y A_s - 1000 S$$

where `A_s` is the design variable. For each value of `A_s`, FORM computes the failure probability and reliability index.

In [ ]:
def lsf(fy, As, S):
    return fy * As - 1000 * S


def run_reliability(As):
    limit_state = ra.LimitState(lsf)
    model = ra.StochasticModel()
    model.addVariable(ra.Lognormal("fy", 260, 18.2))
    model.addVariable(ra.Constant("As", As))
    model.addVariable(ra.Gumbel("S", 9.5, 1.5))

    options = ra.AnalysisOptions()
    options.setE1(1e-6)
    options.setE2(1e-6)

    form = ra.Form(
        analysis_options=options,
        limit_state=limit_state,
        stochastic_model=model,
    )
    form.run()
    return {"pf": float(np.atleast_1d(form.getFailure())[0]), "beta": form.getBeta()}

In [ ]:
as_values = np.array([70, 75, 80, 85.1, 90, 93, 95, 100], dtype=float)
study = ra.DesignStudy(variable="As", values=as_values, analysis=run_reliability)

reliability = study.evaluate()
reliability

## SWTP and the LQI target

Rackwitz's JCSS table is retained as a 1999 PPPUS$ anchor. For present-day studies, `indexed=True` returns an explicitly indexed value using World Bank GDP per capita PPP factors. This keeps the literature value and the update method visible.

In [ ]:
swtp_anchor = ra.SWTP.from_country("CH")
swtp_indexed = ra.SWTP.from_country("CH", indexed=True)

pd.DataFrame(
    [
        {"basis": "Rackwitz anchor", "value_per_life": swtp_anchor.value_per_life, "price_year": swtp_anchor.price_year},
        {"basis": "GDP PPP indexed", "value_per_life": swtp_indexed.value_per_life, "price_year": swtp_indexed.price_year},
    ]
)

For the Fischer, Barnardo, and Faber target table, define

$$K_1 = \frac{C_1}{\mathrm{SWTP} N_F}$$

where `C1` is the marginal safety cost and `N_F` is the expected number of fatalities conditional on failure. The result is a minimum LQI target reliability.

In [ ]:
expected_fatalities = 12
marginal_safety_cost = 5000

k1 = ra.lqi_k1(
    safety_cost_rate=marginal_safety_cost,
    swtp=swtp_indexed,
    expected_fatalities=expected_fatalities,
)
target = ra.lqi_target_reliability(k1)

pd.DataFrame([target.__dict__])

## Cost-benefit objective and LQI feasibility

The objective below follows the JCSS notebook calculation with a constant annual benefit, construction cost proportional to `A_s`, and failure costs discounted over the service life.

In [ ]:
costs = ra.CostBenefitModel(
    benefit_rate=1.2e4,
    interest_rate=0.02,
    service_life=100,
    construction_cost=lambda As: 5000 * As,
    failure_cost=lambda As: 5000 * As + 12 * 1.8e6 + 3e4,
)

assessment = ra.LQIAssessment(
    study=study,
    costs=costs,
    swtp=swtp_indexed,
    consequence=ra.FatalityConsequence(people_exposed=expected_fatalities),
    target=target,
)

results = assessment.evaluate()
results["construction_cost"] = 5000 * results["As"]
results["jcss_lqi_risk_cost"] = ra.jcss_lqi_risk_cost(
    results["construction_cost"],
    results["pf"],
    swtp_indexed,
    expected_fatalities,
)

results[[
    "As",
    "pf",
    "beta",
    "objective",
    "annualized_safety_cost",
    "target_pf",
    "lqi_acceptable",
    "jcss_lqi_risk_cost",
]]

In [ ]:
economic_best = results.loc[results["objective"].idxmax()]
feasible_best = results.loc[results[results["lqi_acceptable"]]["objective"].idxmax()]

pd.DataFrame(
    [
        {"selection": "economic optimum", "As": economic_best["As"], "pf": economic_best["pf"], "objective": economic_best["objective"], "lqi_acceptable": economic_best["lqi_acceptable"]},
        {"selection": "best LQI-feasible", "As": feasible_best["As"], "pf": feasible_best["pf"], "objective": feasible_best["objective"], "lqi_acceptable": feasible_best["lqi_acceptable"]},
    ]
)

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(results["As"], results["objective"], marker="o", label="objective")
ax1.set_xlabel("Reinforcement area As")
ax1.set_ylabel("Objective")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.semilogy(results["As"], results["pf"], marker="s", color="tab:red", label="failure probability")
ax2.axhline(target.pf, color="tab:red", linestyle="--", label="LQI target")
ax2.set_ylabel("Failure probability")

lines = ax1.get_lines() + ax2.get_lines()
labels = [line.get_label() for line in lines]
ax1.legend(lines, labels, loc="best")
fig.tight_layout()

## Canonical JCSS marginal check

The JCSS LQI acceptance condition can also be checked directly when a differentiable cost and failure-rate model are available:

$$\frac{dC(p)}{dp} + \mathrm{SWTP} N_F \frac{dh(p)}{dp} \ge 0$$

The helper below returns this margin. Positive values satisfy the marginal LQI condition.

In [ ]:
from scipy.stats import norm


def jcss_lognormal_pf(p):
    vr = 0.2
    vs = 0.3
    numerator = np.log(p * np.sqrt((1 + vs**2) / (1 + vr**2)))
    denominator = np.sqrt(np.log((1 + vr**2) * (1 + vs**2)))
    return norm.cdf(-numerator / denominator)


def jcss_cost(p):
    return 1e6 + 1e4 * p**1.25


pd.DataFrame(
    {
        "p": [3.0, 3.61, 4.4],
        "pf": [jcss_lognormal_pf(p) for p in [3.0, 3.61, 4.4]],
        "lqi_margin": [
            ra.jcss_lqi_acceptability(jcss_cost, jcss_lognormal_pf, p, 5e6, 10)
            for p in [3.0, 3.61, 4.4]
        ],
    }
)

## Generalized network risk results

For a bridge network or other system, the reliability calculation may produce joint failure scenarios instead of one component failure probability. The LQI layer only needs expected annual risk quantities. Each row below can represent a correlated joint state from an upstream hazard/reliability model, and the economic loss can already include nonlinear network effects such as detours, closures, and repair staging.

In [ ]:
network_scenarios = pd.DataFrame(
    {
        "state": ["Bridge A", "Bridge B", "A and B"],
        "probability": [2.0e-3, 1.5e-3, 2.0e-4],
        "fatalities": [0.2, 0.3, 2.0],
        "economic_loss": [1.0e6, 1.5e6, 8.0e6],
        "failure": [True, True, True],
    }
)

network_risk = ra.RiskResult.from_scenarios(network_scenarios, failure_col="failure")
network_risk.to_dict()

In [ ]:
pd.DataFrame(
    [
        {
            "expected_economic_loss": network_risk.expected_economic_loss,
            "life_safety_cost": network_risk.life_safety_cost(swtp_indexed),
            "total_risk_cost": network_risk.total_risk_cost(swtp_indexed),
            "jcss_cost_with_intervention": ra.jcss_lqi_risk_cost_from_result(
                safety_cost=100_000,
                risk=network_risk,
                swtp=swtp_indexed,
                include_economic_loss=True,
            ),
        }
    ]
)